# Optimización de Modelos Conjunto Soleado por GMM

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [4]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [5]:
datos_dia = datos[datos["Cluster GMM"] == "Lluvioso"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
30,2022-09-02 06:00:00,0.000000,17,91,0,6,Soleado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,94,0,7,Soleado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,97,0,8,Soleado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,93,1,9,Soleado,Lluvioso,438.814997,7720.582326
34,2022-09-02 10:00:00,5030.740421,18,85,2,10,Soleado,Lluvioso,5908.000884,9433.109309
35,2022-09-02 11:00:00,17036.043251,20,71,2,11,Soleado,Lluvioso,5030.740421,22189.147406
44,2022-09-02 20:00:00,2370.417542,24,52,0,20,Soleado,Lluvioso,11972.590689,3411.083740
45,2022-09-02 21:00:00,182.435112,22,61,0,21,Soleado,Lluvioso,2370.417542,382.187718
54,2022-09-03 06:00:00,0.000000,18,87,0,6,Soleado,Lluvioso,0.000000,0.000000
55,2022-09-03 07:00:00,13.512200,18,87,0,7,Soleado,Lluvioso,0.000000,0.000000


In [6]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [7]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
30,17,91,0,6,0.000000,0.000000
31,17,94,0,7,0.000000,6.584959
32,16,97,0,8,0.000000,560.422022
33,17,93,1,9,438.814997,7720.582326
34,18,85,2,10,5908.000884,9433.109309
...,...,...,...,...,...,...
18273,14,87,1,8,67.000000,7302.000000
18274,15,83,2,9,7356.000000,18014.000000
18275,17,71,4,10,17638.000000,23010.000000
18276,19,60,5,11,23339.000000,26156.000000


In [8]:
y = datos_dia[['Generación']]
y

,Generación
30,0.000000
31,0.000000
32,438.814997
33,5908.000884
34,5030.740421
...,...
18273,7356.000000
18274,17638.000000
18275,23339.000000
18276,26323.000000


Dividimos entrenamiento, validación y prueba

In [9]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [10]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 3882, y_train: 3882
X_val: 832, y_val: 832
X_test: 832, y_test: 832


## Escalar con MinMaxScaler

In [11]:
from sklearn.preprocessing import MinMaxScaler

In [12]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [13]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[6.53846154e-01 8.67647059e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [6.53846154e-01 9.11764706e-01 0.00000000e+00 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [6.15384615e-01 9.55882353e-01 0.00000000e+00 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [4.61538462e-01 6.47058824e-01 2.85714286e-01 2.66666667e-01
  2.05400000e-01 5.44266667e-01]
 [6.15384615e-01 3.82352941e-01 2.85714286e-01 3.33333333e-01
  5.85333333e-01 6.35166667e-01]
 [3.84615385e-01 7.20588235e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]]
(3882, 6)


In [14]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.653846,0.867647,0.000000,0.000000,0.000000,0.000000
31,0.653846,0.911765,0.000000,0.066667,0.000000,0.000219
32,0.615385,0.955882,0.000000,0.133333,0.000000,0.018681
33,0.653846,0.897059,0.142857,0.200000,0.014627,0.257353
34,0.692308,0.779412,0.285714,0.266667,0.196933,0.314437
...,...,...,...,...,...,...
12201,0.269231,0.985294,0.000000,0.133333,0.000000,0.000867
12202,0.346154,0.852941,0.142857,0.200000,0.002433,0.050100
12203,0.461538,0.647059,0.285714,0.266667,0.205400,0.544267
12204,0.615385,0.382353,0.285714,0.333333,0.585333,0.635167


In [15]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.38461538 0.76470588 0.         0.06666667 0.         0.        ]
 [0.30769231 0.86764706 0.         0.13333333 0.         0.00243333]
 [0.38461538 0.75       0.14285714 0.2        0.00126667 0.2054    ]
 ...
 [0.76923077 0.64705882 0.57142857 0.26666667 0.82916667 0.91103333]
 [0.92307692 0.38235294 0.14285714 0.86666667 0.6946     0.19466667]
 [0.88461538 0.45588235 0.         0.93333333 0.36236667 0.0164    ]]
(832, 6)


In [16]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
12224,0.384615,0.764706,0.000000,0.066667,0.000000,0.000000
12225,0.307692,0.867647,0.000000,0.133333,0.000000,0.002433
12226,0.384615,0.750000,0.142857,0.200000,0.001267,0.205400
12227,0.500000,0.558824,0.142857,0.266667,0.083400,0.585333
12228,0.615385,0.352941,0.285714,0.333333,0.585333,0.635167
...,...,...,...,...,...,...
15897,0.692308,0.867647,0.285714,0.133333,0.077467,0.516633
15898,0.730769,0.779412,0.428571,0.200000,0.517867,0.824900
15899,0.769231,0.647059,0.571429,0.266667,0.829167,0.911033
15908,0.923077,0.382353,0.142857,0.866667,0.694600,0.194667


In [17]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.84615385 0.57352941 0.         1.         0.04526667 0.        ]
 [0.65384615 0.92647059 0.         0.         0.         0.        ]
 [0.61538462 1.         0.         0.06666667 0.         0.07746667]
 ...
 [0.65384615 0.57352941 0.57142857 0.26666667 0.58793333 0.767     ]
 [0.73076923 0.41176471 0.71428571 0.33333333 0.77796667 0.87186667]
 [0.76923077 0.32352941 0.         1.         0.         0.        ]]
(832, 6)


In [18]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15910,0.846154,0.573529,0.000000,1.000000,0.045267,0.000000
15919,0.653846,0.926471,0.000000,0.000000,0.000000,0.000000
15920,0.615385,1.000000,0.000000,0.066667,0.000000,0.077467
15921,0.653846,0.882353,0.285714,0.133333,0.077467,0.517867
15922,0.692308,0.779412,0.428571,0.200000,0.546133,0.829167
...,...,...,...,...,...,...
18273,0.538462,0.808824,0.142857,0.133333,0.002233,0.243400
18274,0.576923,0.750000,0.285714,0.200000,0.245200,0.600467
18275,0.653846,0.573529,0.571429,0.266667,0.587933,0.767000
18276,0.730769,0.411765,0.714286,0.333333,0.777967,0.871867


In [19]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [20]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[6.53846154e-01 8.75000000e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [6.53846154e-01 9.16666667e-01 0.00000000e+00 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [6.15384615e-01 9.58333333e-01 0.00000000e+00 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [6.53846154e-01 5.97222222e-01 4.44444444e-01 2.66666667e-01
  5.87933333e-01 7.67000000e-01]
 [7.30769231e-01 4.44444444e-01 5.55555556e-01 3.33333333e-01
  7.77966667e-01 8.71866667e-01]
 [7.69230769e-01 3.61111111e-01 0.00000000e+00 1.00000000e+00
  0.00000000e+00 0.00000000e+00]]
(5546, 6)


In [21]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.653846,0.875000,0.000000,0.000000,0.000000,0.000000
31,0.653846,0.916667,0.000000,0.066667,0.000000,0.000219
32,0.615385,0.958333,0.000000,0.133333,0.000000,0.018681
33,0.653846,0.902778,0.111111,0.200000,0.014627,0.257353
34,0.692308,0.791667,0.222222,0.266667,0.196933,0.314437
...,...,...,...,...,...,...
18273,0.538462,0.819444,0.111111,0.133333,0.002233,0.243400
18274,0.576923,0.763889,0.222222,0.200000,0.245200,0.600467
18275,0.653846,0.597222,0.444444,0.266667,0.587933,0.767000
18276,0.730769,0.444444,0.555556,0.333333,0.777967,0.871867


In [22]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [23]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.58533333]
 [0.63516667]
 [0.        ]]
(3882, 1)


In [24]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
12201,0.002433
12202,0.205400
12203,0.585333
12204,0.635167


In [25]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.00000000e+00]
 [1.26666667e-03]
 [8.34000000e-02]
 [5.85333333e-01]
 [6.35166667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.73333333e-03]
 [2.54233333e-01]
 [7.31700000e-01]
 [7.93966667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [5.70000000e-03]
 [2.57933333e-01]
 [7.31700000e-01]
 [7.93966667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.73333333e-03]
 [2.60933333e-01]
 [7.31700000e-01]
 [1.25666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.73333333e-03]
 [2.60933333e-01]
 [7.31700000e-01]
 [7.87500000e-01]
 [7.60266667e-01]
 [7.31966667e-01]
 [7.01566667e-01]
 [7.05666667e-01]
 [2.96100000e-01]
 [1.41666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.10000000e-03]
 [8.35000000e-02]
 [5.85333333e-01]
 [6.27833333e-01]
 [6.08200000e-01]
 [1.13333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.46666667e-03]
 [8.35000000e-02]
 [5.853333

In [26]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12224,0.000000
12225,0.001267
12226,0.083400
12227,0.585333
12228,0.635167
...,...
15897,0.517867
15898,0.829167
15899,0.915400
15908,0.362367


In [27]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.00000000e+00]
 [0.00000000e+00]
 [7.74666667e-02]
 [5.46133333e-01]
 [8.40400000e-01]
 [9.17433333e-01]
 [4.73666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.25000000e-02]
 [5.39233333e-01]
 [8.24700000e-01]
 [9.11100000e-01]
 [4.57200000e-01]
 [4.96666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.74666667e-02]
 [5.18166667e-01]
 [8.30233333e-01]
 [9.11033333e-01]
 [4.69000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.81666667e-02]
 [5.21533333e-01]
 [8.27466667e-01]
 [9.11033333e-01]
 [9.48200000e-01]
 [5.01733333e-01]
 [3.64933333e-01]
 [3.01000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.21333333e-02]
 [5.18833333e-01]
 [8.27000000e-01]
 [9.11233333e-01]
 [8.55500000e-01]
 [8.21300000e-01]
 [7.80066667e-01]
 [5.63466667e-01]
 [4.56366667e-01]
 [3.40666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.71666667e-02]
 [3.60533333e-01]
 [5.79133333e-01]
 [5.85166667e-01]
 [6.07233333e-01]
 [6.03966667e-01]
 [5.94433333e-01]
 [5.88066667e-01]
 [5.90700000e-01]
 [4.468666

In [28]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15910,0.000000
15919,0.000000
15920,0.077467
15921,0.546133
15922,0.840400
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


In [29]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [30]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.77796667]
 [0.87743333]
 [0.        ]]
(5546, 1)


In [31]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


## Preparación para Redes Neuronales

In [32]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [33]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [34]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (3834, 48, 6), y_train: (3834, 1)
X_val: (784, 48, 6), y_val: (784, 1)
X_test: (784, 48, 6), y_test: (784, 1)


## Optuna

In [35]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.0 MB/s eta 0:00:00


In [36]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [36]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-14 15:19:13,829] A new study created in memory with name: no-name-93f95113-fbc9-4da3-9d8c-c781ebb1e729
[I 2025-03-14 15:19:13,975] Trial 0 finished with value: 0.014307871489720373 and parameters: {'num_leaves': 798, 'subsample': 0.7791961843562533, 'colsample_bytree': 0.7613454883540052, 'min_data_in_leaf': 81}. Best is trial 0 with value: 0.014307871489720373.


[LightGBM] [Warning] min_data_in_leaf is set=81, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=81
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=81, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=81
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000937 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 628
[LightGBM] [Info] Number of data points in the train set: 3882, number of used features: 6
[LightGBM] [Info] Start training from score 0.263711
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

[I 2025-03-14 15:19:14,079] Trial 1 finished with value: 0.015180809857264083 and parameters: {'num_leaves': 226, 'subsample': 0.6926277718711827, 'colsample_bytree': 0.9889008989483162, 'min_data_in_leaf': 74}. Best is trial 0 with value: 0.014307871489720373.
[I 2025-03-14 15:19:14,186] Trial 2 finished with value: 0.013432663209897434 and parameters: {'num_leaves': 248, 'subsample': 0.8474906866403503, 'colsample_bytree': 0.4702903866248259, 'min_data_in_leaf': 73}. Best is trial 2 with value: 0.013432663209897434.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 15:19:14,298] Trial 3 finished with value: 0.013907781766938798 and parameters: {'num_leaves': 482, 'subsample': 0.4708770007763666, 'colsample_bytree': 0.6715781358695263, 'min_data_in_leaf': 45}. Best is trial 2 with value: 0.013432663209897434.
[I 2025-03-14 15:19:14,402] Trial 4 finished with value: 0.013907781766938798 and parameters: {'num_leaves': 420, 'subsample': 0.15435307277769955, 'colsample_bytree': 0.6729627828611447, 'min_data_in_leaf': 45}. Best is trial 2 with value: 0.013432663209897434.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 15:19:19,167] Trial 47 finished with value: 0.013256327159725172 and parameters: {'num_leaves': 622, 'subsample': 0.6966673856354603, 'colsample_bytree': 0.41152235474483756, 'min_data_in_leaf': 90}. Best is trial 43 with value: 0.012674407931316807.
[I 2025-03-14 15:19:19,255] Trial 48 finished with value: 0.013440118827114787 and parameters: {'num_leaves': 567, 'subsample': 0.9118047080105817, 'colsample_bytree': 0.5447359010917345, 'min_data_in_leaf': 98}. Best is trial 43 with value: 0.012674407931316807.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 15:19:19,373] Trial 49 finished with value: 0.013476518442676305 and parameters: {'num_leaves': 725, 'subsample': 0.7977993576752206, 'colsample_bytree': 0.5038877466179145, 'min_data_in_leaf': 88}. Best is trial 43 with value: 0.012674407931316807.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

### Random Forest

In [37]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-14 15:19:19,381] A new study created in memory with name: no-name-c383cc62-0396-4421-80f6-1f809ade8418
[I 2025-03-14 15:19:22,270] Trial 0 finished with value: 0.01650776110691218 and parameters: {'n_estimators': 350, 'max_depth': 20, 'min_samples_split': 12, 'min_samples_leaf': 6, 'bootstrap': True}. Best is trial 0 with value: 0.01650776110691218.
[I 2025-03-14 15:19:24,999] Trial 1 finished with value: 0.016603155418212657 and parameters: {'n_estimators': 350, 'max_depth': 40, 'min_samples_split': 20, 'min_samples_leaf': 5, 'bootstrap': True}. Best is trial 0 with value: 0.01650776110691218.
[I 2025-03-14 15:19:26,179] Trial 2 finished with value: 0.016657011089738792 and parameters: {'n_estimators': 150, 'max_depth': 50, 'min_samples_split': 11, 'min_samples_leaf': 7, 'bootstrap': True}. Best is trial 0 with value: 0.01650776110691218.
[I 2025-03-14 15:19:30,438] Trial 3 finished with value: 0.023381900408717817 and parameters: {'n_estimators': 350, 'max_depth': 25, 'min

Mejores hiperparámetros: {'n_estimators': 500, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 1, 'bootstrap': True}


### CTNET

In [36]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [38]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [38]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-14 16:28:04,932] A new study created in memory with name: no-name-d3c0aef8-79c1-4e82-b4ca-d32af0fb7c59
[I 2025-03-14 16:29:21,592] Trial 0 finished with value: 0.11610317975282669 and parameters: {'head_size': 8, 'num_heads': 5, 'ff_dim': 80, 'num_transformer_blocks': 5, 'mlp_units_1': 512, 'mlp_units_2': 224, 'dropout': 0.3078105325726356, 'mlp_dropout': 0.4960747717423044, 'learning_rate': 0.005130356029434432, 'batch_size': 128}. Best is trial 0 with value: 0.11610317975282669.
[I 2025-03-14 16:30:34,229] Trial 1 finished with value: 0.11615276336669922 and parameters: {'head_size': 4, 'num_heads': 3, 'ff_dim': 64, 'num_transformer_blocks': 5, 'mlp_units_1': 512, 'mlp_units_2': 96, 'dropout': 0.3340061875459706, 'mlp_dropout': 0.4824454186697755, 'learning_rate': 0.0002193228901988514, 'batch_size': 512}. Best is trial 0 with value: 0.11610317975282669.
[I 2025-03-14 16:31:36,275] Trial 2 finished with value: 0.06396869570016861 and parameters: {'head_size': 6, 'num_heads

Mejores hiperparámetros: {'head_size': 5, 'num_heads': 8, 'ff_dim': 128, 'num_transformer_blocks': 4, 'mlp_units_1': 320, 'mlp_units_2': 32, 'dropout': 0.15388192050977845, 'mlp_dropout': 0.45081295525163784, 'learning_rate': 0.0007468797484689935, 'batch_size': 128}


### Forescasting

In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-14 18:11:08,139] A new study created in memory with name: no-name-af06ebbc-93e1-4620-95c1-03ab3bcdbb6f
[I 2025-03-14 18:12:34,243] Trial 9 finished with value: 0.21052005887031555 and parameters: {'filters': 32, 'kernel_size': 4, 'lstm_units_1': 256, 'lstm_units_2': 64, 'lstm_units_3': 16, 'dropout_lstm': 0.48956148406414945, 'dropout_dense': 0.1619453820974536, 'learning_rate': 0.000190889656150763, 'batch_size': 512}. Best is trial 9 with value: 0.21052005887031555.
[I 2025-03-14 18:12:46,177] Trial 12 finished with value: 0.21074865758419037 and parameters: {'filters': 64, 'kernel_size': 5, 'lstm_units_1': 128, 'lstm_units_2': 64, 'lstm_units_3': 64, 'dropout_lstm': 0.4956491639233599, 'dropout_dense': 0.29741410582377004, 'learning_rate': 0.00786968431514747, 'batch_size': 256}. Best is trial 9 with value: 0.21052005887031555.
[I 2025-03-14 18:12:56,719] Trial 13 finished with value: 0.20967039465904236 and parameters: {'filters': 64, 'kernel_size': 4, 'lstm_units_1': 64

Mejores hiperparámetros: {'filters': 128, 'kernel_size': 3, 'lstm_units_1': 256, 'lstm_units_2': 128, 'lstm_units_3': 32, 'dropout_lstm': 0.18434525139222352, 'dropout_dense': 0.25916564521047447, 'learning_rate': 0.004850992789382952, 'batch_size': 128}


### Photovoltaic

In [40]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-14 18:20:41,628] A new study created in memory with name: no-name-82ee582c-733e-4126-aa49-db6a03876297


Epoch 32: early stopping
Restoring model weights from the end of the best epoch: 22.


[I 2025-03-14 18:22:07,856] Trial 6 finished with value: 0.05345393344759941 and parameters: {'filters_1': 64, 'filters_2': 64, 'kernel_size': 5, 'gru_units': 64, 'num_heads': 4, 'dropout_rate': 0.43394269554072107, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.0027064606158211874, 'batch_size': 512}. Best is trial 6 with value: 0.05345393344759941.


Epoch 38: early stopping
Restoring model weights from the end of the best epoch: 28.


[I 2025-03-14 18:22:13,677] Trial 11 finished with value: 0.047406937927007675 and parameters: {'filters_1': 32, 'filters_2': 128, 'kernel_size': 5, 'gru_units': 128, 'num_heads': 2, 'dropout_rate': 0.25757537388176993, 'dense_units_1': 128, 'dense_units_2': 16, 'learning_rate': 0.0025328139310004166, 'batch_size': 512}. Best is trial 11 with value: 0.047406937927007675.


...

Epoch 45: early stopping
Restoring model weights from the end of the best epoch: 35.


[I 2025-03-14 18:29:59,364] Trial 46 finished with value: 0.04519796371459961 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 2, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.410032605999901, 'dense_units_1': 64, 'dense_units_2': 32, 'learning_rate': 0.007291563209273338, 'batch_size': 128}. Best is trial 44 with value: 0.04101428762078285.


Epoch 60: early stopping
Restoring model weights from the end of the best epoch: 50.


[I 2025-03-14 18:30:04,607] Trial 45 finished with value: 0.044571273028850555 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 2, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.41220075369374415, 'dense_units_1': 64, 'dense_units_2': 32, 'learning_rate': 0.007490744896088703, 'batch_size': 128}. Best is trial 44 with value: 0.04101428762078285.


Mejores hiperparámetros: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 2, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.4226674454938969, 'dense_units_1': 64, 'dense_units_2': 32, 'learning_rate': 0.00713646125183064, 'batch_size': 128}
